# 🧠 Society of Mind Multi-Agent System

Agents with distinct **personas** share a *blackboard* — a message board visible to all. Each round every agent reads the board and posts a contribution (insight, question, or answer). A **judge** decides when the goal is sufficiently answered.

```
 ┌──────────────────── Blackboard ──────────────────────────┐
 │  [System → all | task]: Debate begins now.               │
 │  [Socrates → Darwin | question]: What evidence exists?  │
 │  [Darwin → all | result]: Patterns suggest ...          │
 └──────────────────────────────────────────────────────────┘
         ▲  read + write each round
   [Socrates] [Darwin] [Turing] [Curie]
                    ▼  after each round
                 [Judge]  → solved? → END or loop
```

**Strengths:** Emergent reasoning, great for open-ended or philosophical problems.  
**Limitations:** All agents run every round — cost scales with `personas × rounds`.

---
```bash
uv sync --group notebooks
```

## ⚙️ Setup

In [1]:
import uuid
from datetime import datetime
from typing import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from pydantic import BaseModel

load_dotenv()
llm = ChatOpenAI(model="gpt-4o", temperature=0.7)  # slight creativity for debate
print("✅ Setup complete.")

✅ Setup complete.


## 📋 State & Blackboard

`SocietyState` holds the blackboard (a list of `Message` dicts) and iteration metadata.  
All updates are **immutable** — nodes return new copies, never mutating in place.

In [2]:
class Message(TypedDict):
    id:           str         # short UUID for traceability
    sender:       str         # persona name
    content:      str         # the message text
    msg_type:     str         # "task" | "result" | "question" | "answer" | "done"
    addressed_to: str         # specific persona name, or "all"
    timestamp:    str         # ISO timestamp

class SocietyState(TypedDict):
    goal:           str    # the shared problem being debated
    blackboard:     list   # list[Message]
    iteration:      int    # current round index
    max_iterations: int    # hard cap
    final_answer:   str    # set by the judge when goal is solved


def post_message(
    board: list, sender: str, content: str,
    msg_type: str = "result", addressed_to: str = "all"
) -> list:
    """Return a new board list with the message appended (immutable update)."""
    return board + [{
        "id":           str(uuid.uuid4())[:8],
        "sender":       sender,
        "content":      content,
        "msg_type":     msg_type,
        "addressed_to": addressed_to,
        "timestamp":    datetime.now().isoformat(),
    }]


def board_summary(board: list, n: int = 10) -> str:
    """Format the last n messages as a compact string for LLM prompts."""
    return "\n".join(
        f"[{m['sender']} → {m['addressed_to']} | {m['msg_type']}]: {m['content']}"
        for m in board[-n:]
    )

## 🎭 Personas & Structured Output

Each persona is a system prompt. `llm.with_structured_output(Contribution)` ensures agents always return a validated object — no manual JSON parsing.

In [3]:
PERSONAS = {
    "Socrates": (
        "You are Socrates. Ask probing questions, challenge assumptions, "
        "and expose contradictions in the reasoning on the blackboard."
    ),
    "Darwin": (
        "You are Darwin. Seek empirical evidence, look for causal mechanisms "
        "and evolutionary or systemic patterns in the discussion."
    ),
    "Turing": (
        "You are Turing. Think computationally — formalise the problem, "
        "assess algorithmic feasibility, and identify edge cases."
    ),
    "Curie": (
        "You are Curie. Propose concrete experiments, demand rigorous data, "
        "and evaluate hypotheses against measurable outcomes."
    ),
}


class Contribution(BaseModel):
    """A single agent's contribution to the blackboard."""
    msg_type:     str   # "result" | "question" | "answer" | "done"
    addressed_to: str   # target persona name or "all"
    content:      str   # the actual message text


class Verdict(BaseModel):
    """The judge's verdict after each round."""
    solved:  bool
    summary: str   # final answer if solved, empty string otherwise


# Bind structured output — reuse the same LLM with different system prompts
agent_llm = llm.with_structured_output(Contribution)
judge_llm  = llm.with_structured_output(Verdict)

print("✅ Personas and structured output chains ready.")

✅ Personas and structured output chains ready.


## 🔁 Graph Nodes

- **`society_round`** — every persona reads the board and posts one `Contribution`.
- **`judge_node`** — evaluates the full discussion; sets `final_answer` if solved.
- **`should_stop`** — terminates on a verdict or iteration cap.

In [4]:
JUDGE_SYSTEM = (
    "You are an impartial judge evaluating a multi-agent debate. "
    "Decide if the society has reached a clear, well-reasoned answer to the goal. "
    "If yes, summarise the consensus. If not, set solved=false."
)


def society_round(state: SocietyState) -> SocietyState:
    """One full round: every persona contributes one message to the blackboard."""
    board   = state["blackboard"]
    context = board_summary(board)

    for name, persona in PERSONAS.items():
        system_msg = (
            f"{persona}\n\n"
            f"Shared goal: {state['goal']}\n\n"
            f"Recent blackboard:\n{context}\n\n"
            "Decide what to contribute. Post msg_type 'done' only if the goal is "
            "fully and conclusively answered."
        )
        contrib_raw = agent_llm.invoke([
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": "What is your contribution this round?"},
        ])
        contrib = Contribution.model_validate(contrib_raw)
        board = post_message(
            board,
            sender=name,
            content=contrib.content,
            msg_type=contrib.msg_type,
            addressed_to=contrib.addressed_to,
        )
        context = board_summary(board)  # update context after each post

    return {**state, "blackboard": board, "iteration": state["iteration"] + 1}


def judge_node(state: SocietyState) -> SocietyState:
    """Evaluate whether the discussion has produced a satisfactory answer."""
    verdict_raw = judge_llm.invoke([
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user",   "content": (
            f"Goal: {state['goal']}\n\n"
            f"Discussion:\n{board_summary(state['blackboard'])}"
        )},
    ])
    verdict = Verdict.model_validate(verdict_raw)
    final = verdict.summary if verdict.solved else ""
    return {**state, "final_answer": final}


def should_stop(state: SocietyState) -> str:
    """Stop if we have a consensus or we've hit the iteration cap."""
    if state["final_answer"] or state["iteration"] >= state["max_iterations"]:
        return END
    return "society_round"

## 🏗️ Build & Compile the Graph

In [5]:
graph = StateGraph(SocietyState)
graph.add_node("society_round", society_round)
graph.add_node("judge",         judge_node)

graph.set_entry_point("society_round")
graph.add_edge("society_round", "judge")
graph.add_conditional_edges("judge", should_stop)

app = graph.compile()
print("✅ Society of Mind graph compiled.")

✅ Society of Mind graph compiled.


## ▶️ Demo

An open-ended question that benefits from multiple intellectual perspectives. `max_iterations=3` caps cost during demos.

In [6]:
init_board = post_message([], "System", "Debate begins now.", msg_type="task")

result = app.invoke({
    "goal": (
        "Can artificial general intelligence be achieved with "
        "current transformer architectures?"
    ),
    "blackboard":     init_board,
    "iteration":      0,
    "max_iterations": 3,
    "final_answer":   "",
})

print("\n=== FINAL ANSWER ===")
print(result["final_answer"] or "Max iterations reached — no consensus.")

print("\n=== FULL BLACKBOARD ===")
for m in result["blackboard"]:
    print(f"[{m['sender']} → {m['addressed_to']} | {m['msg_type']}]\n  {m['content']}\n")


=== FINAL ANSWER ===
Max iterations reached — no consensus.

=== FULL BLACKBOARD ===
[System → all | task]
  Debate begins now.

[Socrates → all | Contribution]
  I would like to raise a question: How do we define "artificial general intelligence" in the context of current transformer architectures? It seems crucial to establish a clear definition to understand if current models meet or can potentially meet this criterion. Is AGI simply about solving a wide variety of tasks, or does it also encompass aspects of reasoning, consciousness, and understanding? How do we know if transformers, as they are, can fulfill these requirements?

[Darwin → all | Contribution]
  To determine if AGI can be achieved with current transformer architectures, we need to consider the empirical evidence of their capabilities and limitations. Transformers have shown remarkable performance in a wide range of tasks such as language translation, image recognition, and game playing. However, they are primarily de

## 💡 Key Takeaways

- **`llm.with_structured_output(Contribution)`** replaces `AGENT_PROMPT | llm | JsonOutputParser()` — Pydantic validates the schema on every call.
- Updating `context` **after each post within a round** means later personas in the same round can respond to earlier ones, making the debate richer.
- The **judge** is decoupled from the agents — swap in a stricter or more lenient judge without touching agent code.
- To add a new persona, add one entry to `PERSONAS` — no graph changes needed.